# Diagonal integration: [RNA, ATAC]

Diagonal integration combines RNA and ATAC measured in different cells, with no pairing between them. If your RNA and ATAC come from the same cells, as in 10x Multiome, use [the vertical tutorial](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_vertical_rna_atac.ipynb). This tutorial runs iNMF and online_iNMF on `D27mini`, the RNA and ATAC of 3,000 cells of the benchmark dataset `D27` with the pairing left out, then on data in your own format.

Most diagonal methods read ATAC as gene-activity scores, made beforehand with a tool such as Signac or ArchR. GLUE and Seurat_v5 read the peak matrix, and MultiMAP and Seurat_v3 need both. [The folder layout](https://dsichang.github.io/scMultiBench/tutorials/run/#folder-layout) lists each method's ATAC files.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_diagonal_rna_atac.ipynb): sections 1 to 4 take about 6 minutes there.

## 1. Install

The pins keep the numpy and pandas already installed, so Colab needs no restart.

In [ ]:
import numpy, pandas
pins = f"numpy=={numpy.__version__} pandas=={pandas.__version__}"
%pip install -q "multibench-sc>=0.5.2" {pins}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D27mini` (15 MB) once. `mtb.env.install` downloads the environments for iNMF and online_iNMF. The download is 0.9 GB.

In [ ]:
import multibench as mtb
import pandas as pd

pd.set_option("display.max_columns", None)   # show every column of the tables

mtb.data.fetch("D27mini")
METHODS = ["iNMF", "online_iNMF"]
envs = mtb.env.install(METHODS)

## 3. Run the methods

`run` runs each method on `D27mini` and [scores its output](https://dsichang.github.io/scMultiBench/tutorials/evaluate/#what-each-metric-measures). The clustering metrics score how well the embedding's clusters and neighbourhoods match the cell types, and how separate the clusters are. The batch metrics measure how well the RNA and ATAC cells mix. The classification scores measure how well the cell types learned on one of them are predicted in the other. Higher is better for every metric.

<details>
<summary>Details</summary>

`params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_run("out/D27mini")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run("D27mini", "diagonal", methods=METHODS, out_dir="out/D27mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Circle size ranks the rows within each column and colour shows the value scaled per column, so a small gap can look large. `res.summary` has the values.

<details>
<summary>Details</summary>

Metrics are grouped by family: blue for clustering, green for batch correction, pink for classification. Each family starts with an `Overall` bar: each row's mean rank over the family's metrics, scaled so that the lowest gets the shortest bar.

A column with fewer than two different values is drawn grey. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw RNA counts in one `.h5ad` file and ATAC in another, with a cell type for each cell in both. online_iNMF needs about 5,000 cells in total.

Recipes cover [labels in a CSV file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-labels-csv) and [no labels](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-no-labels).

<details>
<summary>Details: export</summary>

The names in quotes say where your file holds each part. `"X"` is the main matrix, `"obsm:atac"` the matrix `obsm["atac"]`, `"layer:counts"` the layer `layers["counts"]`, and `"obs:celltype"` the column `obs["celltype"]`. A wrong name after `obsm:`, `layer:` or `obs:` raises an error that lists the names the file has.

`labels=` needs the column in both files. With cell types for the RNA cells only, leave `labels=` out and follow the recipe for no labels.

The folder stores each matrix dense, so drop the genes or peaks you do not need first.

</details>

The next cell writes two small example files, `my_rna.h5ad` and `my_atac.h5ad`, from 90% of the RNA profiles and 90% of the ATAC profiles of `D27mini`. Skip it for your own data.

In [ ]:
mtb.data.write_example("D27mini")

Read your files. For your own data, change every file name marked `# <-`. On Colab, upload the files first: click the folder icon on the left, then the upload icon.

In [ ]:
import anndata as ad

rna = ad.read_h5ad("my_rna.h5ad")     # <- your RNA file
atac = ad.read_h5ad("my_atac.h5ad")   # <- your ATAC file, other cells
print(rna)
print(atac)

Write the dataset folder. Change each name in quotes marked `# <-` to one the printout above lists for your files, with its prefix, as `"obs:celltype"`.

In [ ]:
NAME = "MYDIAG_RNA_ATAC"   # <- a new name for your own data
mtb.io.export_dataset(rna, "mydata/" + NAME,
                      rna="X",                    # <- where the RNA counts are:
                                                  #    "X" or "layer:counts"
                      atac=atac,                  # the ATAC file, values in X
                      atac_kind="gene_activity",  # <- or "peak"
                      labels="obs:celltype",      # <- the column in both files
                      category="diagonal", overwrite=True)
mtb.scan("mydata/" + NAME, "diagonal", methods=METHODS)

In [ ]:
mine = mtb.run("mydata/" + NAME, "diagonal", methods=METHODS,
               out_dir="out/" + NAME)
mine.summary

In [ ]:
mine.plot()

## 6. Stored results

The package ships stored results for 11 methods on `D28`. `load_results` reads them and `mtb.plot.bubble` draws them, without running anything.

<details>
<summary>Details</summary>

There are no stored results for `D27mini`. `D28` is another benchmark dataset of the same task.

</details>

In [ ]:
stored = mtb.load_results("diagonal", dataset="D28")
mtb.plot.bubble(stored)

## 7. Every method of this task

14 methods run this task. [This notebook](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_diagonal_rna_atac_all.ipynb) runs them all. Its 8 environments download 13.9 GB on a computer without a GPU and 13.2 GB on one with a GPU.

## Troubleshooting

`res.summary[res.summary.status != "scored"]` lists each method that failed or was skipped. Its `reason` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a method times out | raise `timeout=` in `run` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.